In [2]:
import sys
!{sys.executable} -m pip install \
langchain \
langchain-community \
langchain-chroma \
faiss-cpu \
wikipedia \
langchain-google-genai \
google-generativeai

  Using cached langchain-1.2.0-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached wikipedia-1.4.0.tar.gz (27 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached langchain_google_genai-4.1.2-py3-none-any.whl.metadata (2.7 kB)
  Using cached google_generativeai-0.8.6-py3-none-any.whl.metadata (3.9 kB)
  Using cached langgraph-1.0.5-py3-none-any.whl.metadata (7.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached langchain_classic-1.0.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached requests-2.32.5-py3-none-any.whl.m


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import sys
print(sys.executable)

!pip --version


c:\Users\sushil\AppData\Local\Programs\Python\Python312\python.exe
pip 24.0 from C:\Users\sushil\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip (python 3.11)



In [4]:
from langchain_community.retrievers import WikipediaRetriever

In [5]:
retriever=WikipediaRetriever(top_k_results=2,lang='en')

In [6]:
query="the geopolitical impact of the silk road"

docs=retriever.invoke(query)

In [7]:
for i , doc in enumerate(docs):
    print(f"\n----Result {i+1} ----")
    print(f"Content: \n{doc.page_content}...") # truncate


----Result 1 ----
Content: 
The Belt and Road Initiative (BRI or B&R), also known as the One Belt One Road (Chinese: 一带一路; pinyin: Yīdài Yīlù) and sometimes called the New Silk Road, is a global infrastructure and economic development strategy of the government of the People's Republic of China.
The initiative was launched by Chinese Communist Party (CCP) General Secretary Xi Jinping in 2013 while visiting Kazakhstan. It aims to invest in over 150 countries and international organizations through six overland economic corridors and the 21st Century Maritime Silk Road. The BRI is central to Chinese foreign policy, promoting trade connectivity and China's leadership role in global affairs. As of 2024, participating countries account for nearly 75% of the world's population and over half of global GDP. Supporters highlight its potential to boost global trade and growth, particularly in developing countries, while critics raise concerns over environmental impact, human rights, and debt-re

In [14]:
# Vector Store Retriver 
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
documents=[
    Document(page_content="The Silk Road was an ancient network of trade routes that connected the East and West. It was central to cultural interaction through regions of the Asian continent connecting the East and West from China to the Mediterranean Sea."),
    Document(page_content="The geopolitical impact of the Silk Road was significant as it facilitated not only trade but also the exchange of culture, religion, and technology between different civilizations."),
    Document(page_content="The Silk Road contributed to the rise and fall of empires, influenced political relationships, and played a crucial role in shaping the modern world."),
    Document(page_content="The Silk Road also had a profound impact on the spread of religions such as Buddhism, Christianity, and Islam, as well as the dissemination of scientific knowledge and technological innovations.")
]

In [16]:
embeddings_model=GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
vectorstore=Chroma.from_documents(documents=documents,embedding=embeddings_model,collection_name="silk_road_docs")

In [17]:
retriever=vectorstore.as_retriever(search_type="similarity",search_kwargs={"k":2})

In [18]:
user_query="the geopolitical impact of the silk road"
results=retriever.invoke(user_query)


In [20]:
for i ,doc in enumerate(results):
    print(f"\n----Result {i+1} ----")
    print(f"Content: \n{doc.page_content}...") # truncate


----Result 1 ----
Content: 
The geopolitical impact of the Silk Road was significant as it facilitated not only trade but also the exchange of culture, religion, and technology between different civilizations....

----Result 2 ----
Content: 
The Silk Road contributed to the rise and fall of empires, influenced political relationships, and played a crucial role in shaping the modern world....


In [35]:
#MMR Retriever means Maximal Marginal Relevance
docs_1=[
    Document(page_content="LangChain is a framework for developing applications powered by language models."),
    Document(page_content="LangChain enables developers to build applications that can understand and generate human-like text."),  
    Document(page_content="With LangChain, you can create chatbots, question-answering systems, and more."),
    Document(page_content="LangChain provides tools for integrating language models with external data sources and APIs."),
    Document(page_content="The LangChain community is active and provides support for developers using the framework.")   
]

In [42]:
from langchain_community.vectorstores import FAISS
embeddings_model=GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
vectorstore=FAISS.from_documents(documents=docs_1,embedding=embeddings_model)

In [37]:
retriever=vectorstore.as_retriever(
    search_type="mmr",  #----- Maximal Marginal Relevance
    search_kwargs={"k":2,"lambda_mult":0.5})  # K is number of docs to return, lambda_mult controls diversity

In [38]:
user_query_1="Virat Kohli's contributions to cricket"
results=retriever.invoke(user_query_1)

In [39]:
for i , doc in enumerate(results):
    print(f"\n----Result {i+1} ----")
    print(f"Content: \n{doc.page_content}...") # truncate


----Result 1 ----
Content: 
The LangChain community is active and provides support for developers using the framework....

----Result 2 ----
Content: 
LangChain provides tools for integrating language models with external data sources and APIs....


In [50]:
import sys
!{sys.executable} -m pip install --upgrade langchain langchain-core langchain-community



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [51]:
#Multi Query Retriver

from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain.retrievers.multi_query import MultiQueryRetriever



ModuleNotFoundError: No module named 'langchain.retrievers'

from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain.retrievers.multi_query import MultiQueryRetriever